# LockForge: Hardware IP Protection via Logic Locking
## LLM4ChipDesign Undergraduate Course Tutorial

This notebook provides a comprehensive introduction to hardware IP protection using logic locking techniques. We'll explore multiple state-of-the-art locking methods implemented in the LockForge framework.

### Learning Objectives
- Understand the fundamentals of logic locking for hardware IP protection
- Implement various locking techniques (AutoLock, CAS-Lock, SFLL-HD, etc.)
- Analyze the security and overhead of different locking methods
- Work with ISCAS benchmark circuits

### Prerequisites
- Basic understanding of digital logic circuits
- Python programming fundamentals
- Familiarity with graph theory concepts (helpful but not required)

## 1. Setup and Installation

First, let's set up our environment and download the LockForge repository.

In [ ]:
# Install required dependencies
!pip install networkx matplotlib numpy pandas --quiet

print("✓ Dependencies installed successfully!")

In [ ]:
# Clone the LockForge repository
import os

if not os.path.exists('LockForge'):
    !git clone https://github.com/DfX-NYUAD/LockForge.git
    print("✓ LockForge repository cloned successfully!")
else:
    print("✓ LockForge repository already exists!")

# Change to the repository directory
%cd LockForge

## 2. Logic Locking Fundamentals

### What is Logic Locking?

Logic locking is a hardware IP protection technique that modifies a circuit design by inserting additional key-controlled logic gates. The circuit only functions correctly when the correct key is applied.

### Key Concepts:

1. **Key Gates**: XOR/XNOR gates inserted into the circuit
2. **Key Inputs**: Additional inputs that control the key gates
3. **Corruptibility**: The ability of incorrect keys to produce wrong outputs
4. **Output Corruption**: Percentage of outputs affected by wrong keys

### Common Locking Techniques:

| Technique | Key Feature | Use Case |
|-----------|-------------|----------|
| **AutoLock** | Genetic Algorithm-based D-MUX insertion | High corruptibility |
| **CAS-Lock** | Corrupt-and-Set for SAT attack resistance | SAT-resistant |
| **SFLL-HD** | Strong logic locking with high HD | Balanced security |
| **SARO** | Stripped functionality logic locking | Anti-SAT protection |
| **TriLock** | Three-stage locking approach | Multi-layer security |

### The BENCH Format

ISCAS benchmark circuits use the `.bench` format:
```
INPUT(A)
INPUT(B)
OUTPUT(Z)
wire1 = AND(A, B)
Z = OR(wire1, C)
```

## 3. Exploring LockForge Structure

In [ ]:
# List available locking techniques
import os
import glob

techniques = [d for d in os.listdir('.') if os.path.isdir(d) and not d.startswith('.')]
techniques = [t for t in techniques if t not in ['Reference_Codes']]

print("Available Logic Locking Techniques:")
print("=" * 50)
for i, tech in enumerate(sorted(techniques), 1):
    print(f"{i}. {tech}")
    
    # Check for README files
    readme_files = glob.glob(f"{tech}/README*.md")
    if readme_files:
        print(f"   📄 Documentation: {os.path.basename(readme_files[0])}")
    
    # Check for locked circuits
    locked_dirs = glob.glob(f"{tech}/*[Ll]ocked*")
    if locked_dirs:
        num_circuits = len(glob.glob(f"{locked_dirs[0]}/*.bench"))
        print(f"   🔒 Pre-locked circuits: {num_circuits}")
    print()

## 4. BENCH File Parser and Utilities

Let's create utilities to work with BENCH files:

In [ ]:
import re
from typing import Dict, List, Tuple

class BenchParser:
    """Parser for ISCAS .bench format files"""
    
    def __init__(self, filepath):
        self.filepath = filepath
        self.inputs = []
        self.outputs = []
        self.gates = {}
        self._parse()
    
    def _parse(self):
        """Parse the bench file"""
        gate_re = re.compile(r'^\s*([A-Za-z0-9_]+)\s*=\s*([A-Za-z0-9_]+)\s*\(\s*([A-Za-z0-9_,\s]+)\s*\)\s*$')
        in_re = re.compile(r'^\s*INPUT\s*\(\s*([A-Za-z0-9_]+)\s*\)\s*$', re.IGNORECASE)
        out_re = re.compile(r'^\s*OUTPUT\s*\(\s*([A-Za-z0-9_]+)\s*\)\s*$', re.IGNORECASE)
        
        with open(self.filepath, 'r') as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#'):
                    continue
                
                # Check for INPUT
                m = in_re.match(line)
                if m:
                    self.inputs.append(m.group(1))
                    continue
                
                # Check for OUTPUT
                m = out_re.match(line)
                if m:
                    self.outputs.append(m.group(1))
                    continue
                
                # Check for gate
                m = gate_re.match(line)
                if m:
                    output = m.group(1)
                    gate_type = m.group(2).upper()
                    inputs = [s.strip() for s in m.group(3).split(',')]
                    self.gates[output] = (gate_type, inputs)
    
    def get_statistics(self):
        """Get circuit statistics"""
        gate_types = {}
        for gate_type, _ in self.gates.values():
            gate_types[gate_type] = gate_types.get(gate_type, 0) + 1
        
        return {
            'num_inputs': len(self.inputs),
            'num_outputs': len(self.outputs),
            'num_gates': len(self.gates),
            'gate_types': gate_types
        }
    
    def visualize_stats(self):
        """Print formatted statistics"""
        stats = self.get_statistics()
        print(f"Circuit Statistics for: {os.path.basename(self.filepath)}")
        print("=" * 60)
        print(f"Primary Inputs:  {stats['num_inputs']}")
        print(f"Primary Outputs: {stats['num_outputs']}")
        print(f"Total Gates:     {stats['num_gates']}")
        print("\nGate Distribution:")
        for gate_type, count in sorted(stats['gate_types'].items()):
            print(f"  {gate_type:12s}: {count:5d} ({100*count/stats['num_gates']:.1f}%)")

# Test the parser
print("BenchParser class defined successfully!")
print("\nExample usage:")
print("parser = BenchParser('path/to/circuit.bench')")
print("parser.visualize_stats()")

## 5. Analyzing Example Circuits

Let's examine some pre-locked circuits to understand the locking techniques:

In [ ]:
# Find and analyze AutoLock circuits
import glob

autolock_circuits = glob.glob('AutoLock/AutoLocked/*.bench')

if autolock_circuits:
    print(f"Found {len(autolock_circuits)} AutoLock pre-locked circuits\n")
    
    # Analyze the first circuit
    sample_circuit = autolock_circuits[0]
    print(f"Analyzing: {os.path.basename(sample_circuit)}\n")
    
    parser = BenchParser(sample_circuit)
    parser.visualize_stats()
    
    # Check for KEYINPUT gates
    key_inputs = [inp for inp in parser.inputs if 'KEY' in inp.upper()]
    if key_inputs:
        print(f"\n🔑 Key Bits: {len(key_inputs)}")
        print(f"   Key Input Names: {', '.join(key_inputs[:5])}{'...' if len(key_inputs) > 5 else ''}")
else:
    print("No AutoLock circuits found. Please check the directory structure.")

## 6. AutoLock: Genetic Algorithm-Based Locking

AutoLock uses a genetic algorithm to select optimal D-MUX insertion sites for maximum output corruptibility.

### How AutoLock Works:

1. **Candidate Enumeration**: Find all valid D-MUX insertion points
2. **Genetic Algorithm**: Optimize site selection for high corruption
3. **MUX Insertion**: Insert 2:1 MUXes controlled by key bits
4. **Key Generation**: Generate and store the correct key

### Parameters:
- `--k`: Number of key bits (default: 64)
- `--pop`: GA population size (default: 200)
- `--gens`: GA generations (default: 100)
- `--seed`: Random seed for reproducibility

In [ ]:
# Example: Running AutoLock on a small benchmark circuit
# Note: This requires having a .bench file to work with

# First, let's check if we have the AutoLock script
if os.path.exists('AutoLock/AutoLock.py'):
    print("AutoLock script found!\n")
    
    # Display the help message
    print("AutoLock Command Line Interface:")
    print("=" * 60)
    !python AutoLock/AutoLock.py --help
else:
    print("AutoLock script not found in expected location.")

# Example command (commented out - uncomment and modify with actual circuit):
# !python AutoLock/AutoLock.py --in circuits/c432.bench --k 32 --pop 100 --gens 50 --seed 42 --out-prefix c432_locked

## 7. CAS-Lock: Corruption and Set for SAT Resistance

CAS-Lock (Corrupt And Set) provides resistance against SAT-based attacks by carefully selecting key gate locations.

### Key Features:
- Protects against SAT attacks
- Uses both corrupt and set functions
- Maintains low area overhead

### CAS-Lock Strategy:
1. **Corrupt Function**: Creates wrong outputs for incorrect keys
2. **Set Function**: Ensures correct output for the correct key
3. **Protected Points**: Strategically chosen insertion points

In [ ]:
# Explore CAS-Lock circuits
cas_circuits = glob.glob('CAS_Lock/Cas_Locked/*.bench')

if cas_circuits:
    print(f"Found {len(cas_circuits)} CAS-Lock circuits\n")
    
    # Compare original and locked circuit (if we can find pairs)
    sample = cas_circuits[0]
    print(f"Sample CAS-Lock circuit: {os.path.basename(sample)}\n")
    
    parser = BenchParser(sample)
    parser.visualize_stats()
    
    # Look for the key file
    key_file = 'CAS_Lock/Cas_Locked/locked_keys.txt'
    if os.path.exists(key_file):
        print("\n📋 Key information available in:", key_file)
        with open(key_file, 'r') as f:
            lines = f.readlines()[:5]
            print("First few lines:")
            for line in lines:
                print(" ", line.strip())
else:
    print("No CAS-Lock circuits found.")

## 8. Comparing Locking Techniques

Let's create a comparison framework to evaluate different locking techniques:

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

class LockingComparison:
    """Compare different logic locking techniques"""
    
    def __init__(self):
        self.results = []
    
    def analyze_locked_circuit(self, circuit_path, technique_name):
        """Analyze a locked circuit and extract metrics"""
        parser = BenchParser(circuit_path)
        stats = parser.get_statistics()
        
        # Count key inputs
        key_inputs = sum(1 for inp in parser.inputs if 'KEY' in inp.upper())
        
        # Count key gates (XOR/XNOR typically used for locking)
        key_gates = sum(1 for gate_type, _ in parser.gates.values() 
                       if gate_type in ['XOR', 'XNOR', 'MUX'])
        
        result = {
            'technique': technique_name,
            'circuit': os.path.basename(circuit_path),
            'key_bits': key_inputs,
            'total_gates': stats['num_gates'],
            'key_gates': key_gates,
            'overhead': (key_gates / stats['num_gates'] * 100) if stats['num_gates'] > 0 else 0
        }
        
        self.results.append(result)
        return result
    
    def scan_technique(self, base_path, technique_name):
        """Scan all circuits for a given technique"""
        circuits = glob.glob(f"{base_path}/*.bench")
        print(f"Scanning {technique_name}: found {len(circuits)} circuits")
        
        for circuit in circuits:
            try:
                self.analyze_locked_circuit(circuit, technique_name)
            except Exception as e:
                print(f"  Error analyzing {os.path.basename(circuit)}: {e}")
    
    def get_dataframe(self):
        """Get results as a pandas DataFrame"""
        return pd.DataFrame(self.results)
    
    def plot_comparison(self):
        """Create visualization of comparison results"""
        df = self.get_dataframe()
        
        if len(df) == 0:
            print("No data to plot")
            return
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # Plot 1: Key bits by technique
        df.groupby('technique')['key_bits'].mean().plot(kind='bar', ax=axes[0], color='steelblue')
        axes[0].set_title('Average Key Bits by Technique')
        axes[0].set_ylabel('Number of Key Bits')
        axes[0].set_xlabel('Locking Technique')
        axes[0].tick_params(axis='x', rotation=45)
        
        # Plot 2: Area overhead by technique
        df.groupby('technique')['overhead'].mean().plot(kind='bar', ax=axes[1], color='coral')
        axes[1].set_title('Average Area Overhead by Technique')
        axes[1].set_ylabel('Overhead (%)')
        axes[1].set_xlabel('Locking Technique')
        axes[1].tick_params(axis='x', rotation=45)
        
        plt.tight_layout()
        plt.show()

# Create comparison object
comparison = LockingComparison()
print("LockingComparison framework created!")

In [ ]:
# Run comparison across multiple techniques
techniques_to_compare = [
    ('AutoLock/AutoLocked', 'AutoLock'),
    ('CAS_Lock/Cas_Locked', 'CAS-Lock'),
    ('SFLL_HD/SFLL_Locked', 'SFLL-HD'),
]

for path, name in techniques_to_compare:
    if os.path.exists(path):
        comparison.scan_technique(path, name)
    else:
        print(f"Path not found: {path}")

# Display results
df_results = comparison.get_dataframe()
if len(df_results) > 0:
    print("\nComparison Results:")
    print("=" * 80)
    print(df_results.to_string(index=False))
    print("\nSummary Statistics:")
    print(df_results.groupby('technique').agg({
        'key_bits': ['mean', 'min', 'max'],
        'overhead': ['mean', 'min', 'max']
    }).round(2))
    
    # Plot comparison
    comparison.plot_comparison()
else:
    print("No circuits found to compare. Please check the directory structure.")

## 9. Security Analysis Framework

Understanding the security of logic locking requires analyzing attack resistance:

In [ ]:
import random

class SecurityAnalyzer:
    """Analyze security properties of locked circuits"""
    
    def __init__(self, circuit_path):
        self.parser = BenchParser(circuit_path)
        self.circuit_path = circuit_path
    
    def estimate_key_space(self):
        """Estimate the size of the key space"""
        key_inputs = sum(1 for inp in self.parser.inputs if 'KEY' in inp.upper())
        return 2 ** key_inputs, key_inputs
    
    def analyze_key_distribution(self):
        """Analyze how keys are distributed in the circuit"""
        # Find which gates use key inputs
        key_usage = {}
        key_inputs = set(inp for inp in self.parser.inputs if 'KEY' in inp.upper())
        
        for gate_out, (gate_type, gate_ins) in self.parser.gates.items():
            keys_used = [inp for inp in gate_ins if inp in key_inputs]
            if keys_used:
                key_usage[gate_out] = {
                    'type': gate_type,
                    'keys': keys_used
                }
        
        return key_usage
    
    def report(self):
        """Generate security analysis report"""
        print(f"Security Analysis: {os.path.basename(self.circuit_path)}")
        print("=" * 70)
        
        # Key space
        key_space, num_keys = self.estimate_key_space()
        print(f"\n🔑 Key Space Analysis:")
        print(f"   Number of key bits: {num_keys}")
        print(f"   Total key combinations: 2^{num_keys} = {key_space:,}")
        print(f"   Brute force complexity: O(2^{num_keys})")
        
        # Key distribution
        key_usage = self.analyze_key_distribution()
        print(f"\n🔒 Key Distribution:")
        print(f"   Gates using keys: {len(key_usage)}")
        
        # Gate types using keys
        gate_types = {}
        for info in key_usage.values():
            gt = info['type']
            gate_types[gt] = gate_types.get(gt, 0) + 1
        
        print(f"   Key gate types:")
        for gt, count in sorted(gate_types.items()):
            print(f"      {gt}: {count}")
        
        # Security metrics
        stats = self.parser.get_statistics()
        lock_ratio = len(key_usage) / stats['num_gates'] * 100
        print(f"\n📊 Security Metrics:")
        print(f"   Locking ratio: {lock_ratio:.2f}% of gates")
        print(f"   Average keys per locked gate: {num_keys / len(key_usage):.2f}")

# Example usage
print("SecurityAnalyzer class created!")
print("\nUsage example:")
print("analyzer = SecurityAnalyzer('path/to/locked_circuit.bench')")
print("analyzer.report()")

In [ ]:
# Analyze security of sample circuits
sample_circuits = []

# Collect one sample from each technique
for path, name in techniques_to_compare:
    if os.path.exists(path):
        circuits = glob.glob(f"{path}/*.bench")
        if circuits:
            sample_circuits.append((circuits[0], name))

# Analyze each sample
for circuit_path, technique in sample_circuits:
    print(f"\n{'='*70}")
    print(f"Technique: {technique}")
    print(f"{'='*70}")
    analyzer = SecurityAnalyzer(circuit_path)
    analyzer.report()
    print()

## 10. Circuit Visualization

Let's create a simple visualization of circuit structure:

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

def visualize_circuit_graph(bench_path, max_nodes=50):
    """Create a graph visualization of the circuit"""
    parser = BenchParser(bench_path)
    
    # Create directed graph
    G = nx.DiGraph()
    
    # Add nodes and edges
    for output, (gate_type, inputs) in parser.gates.items():
        G.add_node(output, type='gate', gate_type=gate_type)
        for inp in inputs:
            G.add_edge(inp, output)
    
    # Add input and output nodes
    for inp in parser.inputs:
        G.add_node(inp, type='input')
    for out in parser.outputs:
        if out in G:
            G.nodes[out]['type'] = 'output'
    
    # Limit visualization size
    if len(G.nodes()) > max_nodes:
        print(f"Circuit too large ({len(G.nodes())} nodes). Sampling {max_nodes} nodes...")
        # Sample nodes
        nodes_to_keep = set(parser.inputs[:10] + parser.outputs[:10])
        nodes_to_keep.update(random.sample(list(parser.gates.keys()), 
                                          min(max_nodes - len(nodes_to_keep), len(parser.gates))))
        G = G.subgraph(nodes_to_keep).copy()
    
    # Color nodes by type
    color_map = []
    for node in G.nodes():
        node_type = G.nodes[node].get('type', 'gate')
        if node_type == 'input':
            if 'KEY' in node.upper():
                color_map.append('red')  # Key inputs in red
            else:
                color_map.append('lightblue')  # Regular inputs
        elif node_type == 'output':
            color_map.append('lightgreen')
        else:
            color_map.append('lightgray')
    
    # Create layout
    plt.figure(figsize=(12, 8))
    pos = nx.spring_layout(G, seed=42, k=1, iterations=50)
    
    # Draw
    nx.draw(G, pos, node_color=color_map, with_labels=True, 
            node_size=500, font_size=8, font_weight='bold',
            arrows=True, edge_color='gray', alpha=0.7)
    
    plt.title(f"Circuit Graph: {os.path.basename(bench_path)}\n"
             f"(Red=Key Inputs, Blue=Inputs, Green=Outputs, Gray=Gates)")
    plt.tight_layout()
    plt.show()
    
    print(f"\nGraph Statistics:")
    print(f"  Nodes: {len(G.nodes())}")
    print(f"  Edges: {len(G.edges())}")
    print(f"  Avg degree: {sum(dict(G.degree()).values()) / len(G.nodes()):.2f}")

# Visualize a sample circuit
if sample_circuits:
    circuit_to_viz = sample_circuits[0][0]
    print(f"Visualizing: {os.path.basename(circuit_to_viz)}\n")
    visualize_circuit_graph(circuit_to_viz)
else:
    print("No circuits available for visualization.")

## 11. Hands-On Exercises

### Exercise 1: Key Strength Analysis
Compare the key space of different locking techniques and discuss the trade-offs between security and overhead.

### Exercise 2: Custom Locking
Modify the AutoLock parameters to create a circuit with:
- 32-bit key
- Population size of 150
- 75 generations

Compare the results with default parameters.

### Exercise 3: Attack Resistance
Research different attacks on logic locking (SAT attack, AppSAT, removal attack) and identify which techniques provide the best resistance.

### Exercise 4: Overhead Analysis
Calculate the area and power overhead for different locking techniques. Which technique provides the best security-to-overhead ratio?

In [ ]:
# Exercise workspace
# Students can use this cell for their exercises

print("Exercise Workspace")
print("=" * 60)
print("Use this cell to complete the exercises above.")
print("\nExample: Analyzing overhead")
print(""")
# Your code here
# Example:
# circuit = 'AutoLock/AutoLocked/c1355_locked.bench'
# parser = BenchParser(circuit)
# stats = parser.get_statistics()
# print(f"Total gates: {stats['num_gates']}")
""")

## 12. Advanced Topics

### SAT Attack on Logic Locking

The SAT (Boolean Satisfiability) attack is a powerful technique that can break many logic locking schemes. It works by:
1. Creating a miter circuit between locked and oracle circuits
2. Using a SAT solver to find discriminating input patterns
3. Iteratively eliminating incorrect keys

### Defense Mechanisms

Modern locking techniques defend against SAT attacks through:
- **Corrupt-and-Set (CAS)**: Increases SAT attack iterations
- **Stripped Functionality (SFLL)**: Removes functionality that reveals key
- **Anti-SAT**: Adds specific logic to make SAT attacks infeasible

### Future Directions

1. **Machine Learning Attacks**: Using ML to break logic locking
2. **PUF-Based Locking**: Physical unclonable functions for keys
3. **Hybrid Techniques**: Combining multiple locking approaches
4. **Post-Silicon Activation**: Locking that activates after manufacturing

## 13. Additional Resources

### Research Papers
1. Rajendran et al. "Security Analysis of Logic Obfuscation" (DAC 2012)
2. Subramanyan et al. "Evaluating the Security of Logic Encryption Algorithms" (HOST 2015)
3. Yasin et al. "SARLock: SAT Attack Resistant Logic Locking" (HOST 2016)

### Tools and Frameworks
- **LockForge**: Multi-technique logic locking framework
- **LSS-Bench**: Logic synthesis and security benchmarks
- **ABC**: Berkeley Logic Synthesis Tool

### Online Resources
- ISCAS Benchmark Circuits: Standard test circuits
- HOST Conference: Hardware security conference
- DAC Conference: Design automation conference

### Contact
For questions about this tutorial or LockForge:
- GitHub: https://github.com/DfX-NYUAD/LockForge
- Check the repository for updates and additional examples

## 14. Summary and Conclusions

In this tutorial, we've covered:

✅ **Fundamentals of Logic Locking**
- Basic concepts and terminology
- BENCH file format and parsing
- Circuit analysis techniques

✅ **Locking Techniques**
- AutoLock (GA-based optimization)
- CAS-Lock (SAT-resistant)
- SFLL-HD (Stripped functionality)
- Comparison and trade-offs

✅ **Security Analysis**
- Key space estimation
- Attack resistance evaluation
- Overhead analysis

✅ **Practical Skills**
- Circuit parsing and analysis
- Running locking tools
- Visualizing results
- Comparing techniques

### Key Takeaways

1. **No Perfect Solution**: Each locking technique has trade-offs between security, overhead, and attack resistance.

2. **Attack Evolution**: New attacks constantly emerge, requiring continuous improvement of locking techniques.

3. **Practical Considerations**: Real-world deployment must balance security needs with design constraints.

4. **Active Research Area**: Logic locking remains an active area of research in hardware security.

### Next Steps

- Experiment with different parameters and techniques
- Read recent papers on logic locking advances
- Contribute to open-source hardware security tools
- Explore related topics: PUFs, trojans, side-channel attacks

In [ ]:
# Final setup check and summary
print("LockForge Tutorial Complete! 🎉")
print("=" * 60)
print("\nYou've learned about:")
print("  ✓ Logic locking fundamentals")
print("  ✓ Multiple locking techniques")
print("  ✓ Security analysis methods")
print("  ✓ Circuit analysis and visualization")
print("\nThank you for completing this tutorial!")
print("\nFor questions or contributions, visit:")
print("  https://github.com/DfX-NYUAD/LockForge")